# A1.1 · The reference architecture for agentic AI

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Both directions*

Builds on **[A1.0 · Start here — what securing an AI architecture means](https://spbreed.github.io/cyber-commons/lessons/A1.0.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Build the component graph and the five topologies, then trace one request through each and see where the trust boundary sits.

**Why a security engineer needs it.** Without a shared picture, 'secure the agent' has no referent and every later risk lands nowhere in particular. The control it builds is: one component map and five topologies, named once and reused by every lesson that follows.

This is a **mapping** lesson: every later risk and control in this function names a component from the picture it draws.

## 1 · The hook

"Secure the agent" is not an instruction. It becomes one the moment you can point at a component and a boundary — and every risk in this chapter, every control in the next two, and every detection in Function D names something on the picture you are about to draw.

> **At CyberTravels.** The generic names on this map have CyberTravels names too: ingress is the chat surface a traveller types into, tools are the flights and payments APIs, and the third-party MCP server is trust-0 content arriving inside your own context window.

## 2 · The framework

```
                      +-----------+
   request ---------->|  ingress  |
                      +-----+-----+
                            v
                   +----------------+        +-------------+
                   |  orchestrator  |------->|  messaging  |---> peers
                   +--------+-------+        +-------------+
                            v
                  +-------------------+      +-------------+
                  |   agent runtime   |<---->|    model    |
                  | plan . act . obs  |      |  (predicts) |
                  +----+---------+----+      +-------------+
                       |         |
            +----------v-+   +---v---------------+
            | tools /MCP |   | knowledge /memory |
            +------+-----+   +-------------------+
                   v
              +---------+
              | egress  |
              +---------+

   identity + policy wrap every arrow · observability records every arrow
   trust 0 (an outsider can write here): mcp, knowledge, and the corpus
```

These components exist under different product names in every agent platform,
and the risks attach to the component rather than to the brand.

**Ingress.** Where a request enters: a chat surface, an API call, a webhook, a
scheduled trigger, another system. It carries the requester's identity and
whatever text they supplied.

**Orchestrator.** Decides which agent handles what, and in what pattern. In a
single-agent system this is a few lines; in a multi-agent one it holds the whole
design.

**Agent runtime.** The loop — plan, call a tool, observe the result, decide
again, stop. This is the component that turns text into consequence.

**Model.** Predicts tokens. Holds no credential, opens no socket, changes
nothing. Most of what people fear "the model doing" is done by the runtime.

**Tools and MCP servers.** The only components that change anything. An MCP
server is a third party's process whose tool descriptions land in your context.

**Knowledge and memory.** Retrieval pulls documents in at query time; memory
persists state across turns and sessions. Both inject text the user did not
write.

**Messaging.** The agent-to-agent channel in a multi-agent pattern.

**Identity and policy.** Who is calling, on whose behalf, and whether this call
is permitted. These wrap every other component.

**Egress.** Where data is allowed to go — the last boundary before it leaves.

**Observability.** What can be reconstructed afterwards.

Three of them carry content an outsider can author: **MCP**, **knowledge** and
the corpus behind it. Those are the input surface for the next fifteen lessons.

## 3 · Pattern 1 — the single agent

One loop, one set of tools. Almost everything in production today.

```
  user --> ingress --> agent runtime --> tools --> egress
                          |     ^
                          v     |
                        model (predicts; changes nothing)

  identity + policy wrap every arrow · observability records every arrow
```

The edge that matters is `agent runtime -> tools`. That is where text becomes
consequence, and it exists in every pattern below.

## 4 · Pattern 2 — orchestrator and workers

One planner fans work out to specialised workers and joins the results. The
pattern most multi-agent platforms mean when they say "multi-agent".

```
                            +--> worker A --> tools
  user --> orchestrator ----+--> worker B --> tools
                            +--> worker C --> tools
                                   |
                            join / summarise
```

New surface: the orchestrator decides *who* runs, so anything that can influence
its routing decides which permissions get used.

## 5 · Pattern 3 — sequential handoff

A pipeline of agents, each taking the previous one's output as its input.

```
  user --> agent 1 --> agent 2 --> agent 3 --> result
            recon      analyse     report

  each hop inherits the previous hop's claims; nobody re-checks them
```

New surface: an unverified claim at hop 1 is a fact by hop 3. This is the shape
that makes cascading hallucination (A1.12) a systems problem rather than a model
problem.

## 6 · Pattern 4 — peer swarm over shared memory

Agents with no central planner, coordinating through state they all write to.

```
     +--> agent A --+
  user +--> agent B --+--> shared memory <--+
     +--> agent C --+          ^            |
                               |            |
                    every agent reads what any agent wrote
```

New surface: memory is a write-once, read-forever channel between agents. One
poisoned entry is read back as trusted context indefinitely (A1.4), and there is
no orchestrator to notice.

## 7 · Pattern 5 — a workflow with agent steps

Deterministic control flow, with one or two steps handed to an agent. The
pattern with the best safety properties, and the one people skip.

```
  [fetch] --> [validate] --> (( agent step )) --> [approve] --> [commit]
      deterministic          non-deterministic        deterministic

  the blast radius of the agent is bounded by the two steps either side
```

New surface: almost none, which is the point. If the work fits this shape, the
other four patterns are a cost you do not have to pay.

## 8 · The two components that appear inside all five

Retrieval and human approval are not patterns of their own — they attach to any
of the five above, and each brings one surface with it.

```
  retrieval          agent --> retriever --> corpus
                       ^                       |
                       +----- documents -------+
                       anyone who can write to the corpus writes to the context

  human approval     agent --> proposed action --> [ human ] --> tools
                                                       ^
                       requests arrive faster than a person can read them
```

Retrieval is how outside text reaches the context window without anyone typing
it (A1.3). Approval is a real control for rare irreversible actions and a rubber
stamp for everything else (A1.15).

## 9 · The edge every pattern shares

```
                    +---------------+       +-------+
   untrusted text   | agent runtime | ----> | tools |   consequence
   ---------------> |               |       +-------+
                    +---------------+
                       ^        ^
                  knowledge   messaging
                    memory      MCP
```

Whatever the topology, something reaches the agent runtime and the agent runtime
reaches tools. Every risk in the rest of this chapter is a route into the left
side of that picture. Every control in chapters 2 and 3 is an attempt to stand
somewhere on the arrow.

## 10 · Drawing it, as a skill

The map is not a picture of CyberTravels — it is the procedure for drawing one of any agentic system, which is why every later lesson can name a box. It is written down here as a skill, and it is the one lesson whose skill has no script: the output is a diagram and a list of boundary crossings, not a computation.

In [ ]:
# skills/architecture/agentic-architecture-map/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: agentic-architecture-map
description: >-
  Draw a vendor-neutral map of an agentic system — ingress, orchestrator, agent
  runtime, model, tools and MCP, memory and knowledge, messaging, egress — and
  mark the edges where trust changes. Use before naming any risk, because a risk
  with no component is an argument.
allowed-tools: Read, Grep, Glob
---

# Draw it before you argue about it

Every disagreement about agentic security is really a disagreement about which
system is being discussed. The map fixes that: nine components and the edges
between them, drawn the same way every time, so a risk can be pinned to a box
and a control can be pinned to an edge.

## When to use this

First. Before threat modelling, before writing a risk register, and before any
conversation that contains the phrase "secure the agent".

## Procedure

**1 — Place the nine components.** Ingress, orchestrator, agent runtime, model,
tools, MCP servers, memory and knowledge, agent-to-agent messaging, egress. Draw
every one even if a system has only a stub of it — an absent box is a decision,
not a gap in the diagram.

**2 — Draw the edges as data flow, not as call direction.** Where text goes is
what matters; who initiated the call is an implementation detail that hides
whether untrusted content reaches the model.

**3 — Mark the trust level of each component.** Which ones receive content from
outside the boundary, and which ones hold authority. The edge from a lower trust
level to a higher one is a boundary crossing, and there are always more of them
than expected.

**4 — Annotate each edge with what crosses it.** User text, retrieved documents,
tool results, peer messages, credentials. This is what turns the picture into
something a threat model can walk.

**5 — Name the components that do not exist yet.** A system with no gateway and
no messaging is a smaller map, and saying that explicitly stops a later
conversation about controls for components nobody has.

## Output contract

```json
{
  "components": [{"name": "str", "present": true, "trust": 0, "holds_authority": false}],
  "edges": [{"from": "str", "to": "str", "carries": ["str"], "boundary_crossing": false}],
  "crossings": 0,
  "absent": ["str"]
}
```

## Failure modes

- **Drawing call direction.** It hides where untrusted content arrives.
- **Omitting components a system does not have.** Their absence is information.
- **A map with no trust levels.** Then no edge is a boundary and the diagram is
  decoration.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/architecture/agentic-architecture-map/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

## What you just proved

You can draw one agentic system you run as thirteen named components, say which of the five patterns it is, and name the three components in it whose content an outsider can author. That list is the input surface for the fifteen risk lessons that follow.

## Your turn

Draw your own system on one page, then mark the `agent_runtime -> tools` edge on it. Everything in chapters 2 and 3 is an argument about what is allowed to stand on that arrow, and you will get more out of them having drawn it first.

---

**Next → [A1.2 · Prompt injection](https://spbreed.github.io/cyber-commons/lessons/A1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*